# ZINDI Arabic Multi-Label Classification

## Setup

In [1]:
%pip install -q -U "huggingface-hub>=1.23.0,<2.0" "transformers>=4.48.0" datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.9 MB/s eta 0:00:00


In [2]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 26.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


## Imports

In [3]:
# Standard library imports
import hashlib
import json
import os
import random
import re
import sqlite3
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pydantic import BaseModel, Field
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments

# Google / Gemini specific imports
import google.generativeai as genai
from google import genai as genai_new
from google.genai import types
from google.colab import userdata

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Env Variables

In [12]:
# Regular expression to remove Arabic diacritics (tashkeel)
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061a\u064b-\u0652]")

# Directory where data files are located
DATA_DIR = Path("/content")

# Column name for unique identifiers
ID_COLUMN = "ID"

# Regular expression to match non-alphanumeric and non-Arabic characters
NON_TEXT = re.compile(r"[^\w\s\u0600-\u06ff]")

# List of target columns, excluding 'Qatar Related'
REMAINING_TARGETS = [
    "Geography",
    "Politics & Conflict",
    "Health & Wellbeing",
    "Science",
    "Sports",
]

# Directory for output files (e.g., submissions)
OUTPUT_DIR = Path("/content/submissions")

# Specific target column for Qatar-related classification
QATAR_COLUMN = "Qatar Related"

# Random seed for reproducibility
SEED = 42

# All target columns, including 'Qatar Related'
TARGET_COLUMNS = [QATAR_COLUMN] + REMAINING_TARGETS

# Column name for text content
TEXT_COLUMN = "text"

# Regular expression to normalize whitespace
WHITESPACE = re.compile(r"\s+")

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The specific Gemini model version to be used for LLM classification
GEMINI_MODEL_NAME = "gemini-3.6-flash"

## Utils

In [5]:
def load_data(data_dir=DATA_DIR):
    """
    Loads training and test data from CSV files.

    Args:
        data_dir (Path): The directory containing 'Train.csv' and 'Test.csv'.

    Returns:
        tuple: A tuple containing the train_frame (pd.DataFrame) and test_frame (pd.DataFrame).

    Raises:
        ValueError: If essential columns are missing from the loaded dataframes.
    """
    train_frame = pd.read_csv(data_dir / "Train.csv")
    test_frame = pd.read_csv(data_dir / "Test.csv")

    # Define required columns for validation
    required_train = {ID_COLUMN, TEXT_COLUMN, *TARGET_COLUMNS}
    required_test = {ID_COLUMN, TEXT_COLUMN}

    # Check for missing columns
    missing_train = required_train.difference(train_frame.columns)
    missing_test = required_test.difference(test_frame.columns)

    if missing_train or missing_test:
        raise ValueError(
            f"Missing train columns: {missing_train}; missing test columns: {missing_test}"
        )
    return train_frame, test_frame

In [20]:
def preprocess_arabic(text):
    """
    Applies a series of normalization steps to Arabic text.

    This function removes diacritics, normalizes common Arabic letter variations
    (e.g., 'أ', 'إ', 'آ' to 'ا'), converts 'ى' to 'ي' and 'ة' to 'ه',
    removes non-Arabic, non-alphanumeric characters, and standardizes whitespace.

    Args:
        text (str): The Arabic text to preprocess.

    Returns:
        str: The preprocessed Arabic text.
    """
    text = str(text)
    # Remove diacritics
    text = ARABIC_DIACRITICS.sub("", text)
    # Normalize common Arabic letter variations
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ة", "ه")
    # Remove non-text characters and normalize whitespace
    return WHITESPACE.sub(" ", NON_TEXT.sub(" ", text)).strip()

In [21]:
def score_predictions(actual, predicted, target_columns=TARGET_COLUMNS):
    """
    Calculates weighted F1-score for each target column and returns the mean F1-score.

    Args:
        actual (pd.DataFrame): DataFrame containing the true labels.
        predicted (pd.DataFrame): DataFrame containing the predicted labels.
        target_columns (list): A list of column names for which to calculate F1-scores.

    Returns:
        dict: A dictionary where keys are target column names and 'mean',
              and values are their corresponding weighted F1-scores.
    """
    scores = {
        column: f1_score(actual[column], predicted[column], average="weighted")
        for column in target_columns
    }
    scores["mean"] = float(np.mean(list(scores.values())))
    return scores

In [22]:
def make_submission(test_frame, predictions, output_path):
    """
    Formats predictions into the competition submission format and saves to CSV.

    Args:
        test_frame (pd.DataFrame): The original test dataframe with ID_COLUMN.
        predictions (pd.DataFrame): DataFrame containing the predicted labels for TARGET_COLUMNS.
        output_path (Path): The file path to save the submission CSV.

    Returns:
        pd.DataFrame: The formatted submission DataFrame.
    """
    submission = test_frame[[ID_COLUMN]].copy()
    # Ensure the order of columns matches TARGET_COLUMNS
    submission[TARGET_COLUMNS] = predictions[TARGET_COLUMNS].to_numpy()
    submission.to_csv(output_path, index=False)
    return submission

In [23]:
class NewsDataset(Dataset):
    """
    A custom PyTorch Dataset for handling news article text and their labels.
    It tokenizes text using a pre-trained tokenizer and prepares labels for multi-head classification.
    """

    def __init__(self, frame, is_test=False):
        self.is_test = is_test
        self.encodings = tokenizer(
            frame[TEXT_COLUMN].fillna("").tolist(),
            truncation=True,
            padding="max_length",
            max_length=256,
        )

        if not self.is_test:
            # Only include labels for the 5 targets handled by AraBERT
            self.labels = {
                column: torch.tensor(label_encoders[column].transform(frame[column]))
                for column in REMAINING_TARGETS
            }

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        if not self.is_test:
            item["labels"] = {column: values[index] for column, values in self.labels.items()}
        return item


class MultiHeadAraBERT(torch.nn.Module):
    """
    A multi-head classification model that fine-tunes 5 heads for non-Qatar targets.
    """

    def __init__(self, class_counts):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.encoder.config.hidden_size
        # Heads are created only for REMAINING_TARGETS
        self.heads = torch.nn.ModuleDict({
            column: torch.nn.Linear(hidden_size, count)
            for column, count in class_counts.items()
        })

    def forward(self, input_ids, attention_mask, labels=None):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        logits = {column: head(pooled) for column, head in self.heads.items()}

        if labels is None:
            return logits

        # Loss calculation specifically for the 5 AraBERT heads
        losses = [
            torch.nn.functional.cross_entropy(logits[column], labels[column])
            for column in REMAINING_TARGETS
        ]
        total_loss = torch.stack(losses).mean()
        return (total_loss, logits)

In [24]:
def custom_data_collator(features):
    """
    A custom data collator for the Hugging Face Trainer to handle multi-head labels.
    It stacks input features and, if labels are present and are a dictionary (for multi-head),
    it stacks each target label individually.

    Args:
        features (list): A list of feature dictionaries, where each dictionary
                         represents one sample from the dataset.

    Returns:
        dict: A dictionary containing stacked 'input_ids', 'attention_mask', and 'labels' (if present).
    """
    # Stack input_ids and attention_mask from the batch of features
    batch = {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack(
            [f["attention_mask"] for f in features]
        ),
    }

    # If labels exist in the first feature (assuming all have or none have labels)
    if "labels" in features[0]:
        first_label = features[0]["labels"]

        if isinstance(first_label, dict):
            # If labels are a dictionary (multi-head case), stack each target head individually
            batch["labels"] = {
                key: torch.tensor([f["labels"][key] for f in features])
                for key in first_label.keys()
            }
        else:
            # For standard single-label classification, stack labels directly
            batch["labels"] = torch.tensor([f["labels"] for f in features])

    return batch

In [25]:
def predict_arabert(model, frame, is_test=False):
    """
    Generates predictions for the AraBERT multi-head model on a given dataframe.

    Args:
        model (torch.nn.Module): The trained MultiHeadAraBERT model.
        frame (pd.DataFrame): The DataFrame containing the text data to predict on.
        is_test (bool): Whether the frame is a test set (influences dataset creation).

    Returns:
        pd.DataFrame: A DataFrame containing the inverse-transformed predicted labels
                      for each target column (excluding QATAR_COLUMN).
    """
    # Create a NewsDataset instance for the given frame
    dataset = NewsDataset(frame, is_test=is_test)
    # Create a DataLoader for batch processing
    loader = DataLoader(dataset, batch_size=16, shuffle=False)
    # Determine the device (GPU if available, else CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Move model to the selected device and set to evaluation mode
    model.to(device).eval()
    # Initialize a dictionary to store predictions for each target column
    predictions = {column: [] for column in TARGET_COLUMNS[1:]}

    # Disable gradient calculation during inference for efficiency
    with torch.no_grad():
        for batch in loader:
            # Move input tensors to the correct device
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }

            # Perform forward pass
            out = model(**inputs)
            # Extract logits: handle cases where model outputs a tuple (loss, logits) or just logits dict
            logits = out[1] if isinstance(out, tuple) else out

            # Process predictions for each target column
            for column in TARGET_COLUMNS[1:]:
                # Get the predicted class by finding the argmax of logits, move to CPU, convert to list
                preds = logits[column].argmax(dim=1).cpu().tolist()
                predictions[column].extend(preds)

    # Convert numerical predictions back to original labels using inverse_transform
    return pd.DataFrame({
        column: label_encoders[column].inverse_transform(predictions[column])
        for column in TARGET_COLUMNS[1:]
    })

## Preprocess text and create a validation split

This section applies a specific Arabic normalizer that handles diacritics, punctuation, and letter variations (such as normalizing 'Alif' forms, 'Ya', and 'Ta Marbuta'). After cleaning, the data is split into training and validation sets, using a stratified approach on the `Qatar Related` target to ensure balanced representation across both subsets.

In [13]:
# Load training and test dataframes
train_df, test_df = load_data()
print(f"Train shape: {train_df.shape}; test shape: {test_df.shape}")

# Display the first 2 rows of the training dataframe
display(train_df.head(2))

# Display count of missing values for text and target columns
display(train_df[[TEXT_COLUMN, *TARGET_COLUMNS]].isna().sum().to_frame("missing"))

# Display value counts for each target column to understand distribution
for column in TARGET_COLUMNS:
    print(f"\n{column}")
    display(train_df[column].value_counts(dropna=False).to_frame("count"))

Train shape: (3786, 8); test shape: (1262, 2)


,ID,text,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,fL8IqbB5haFSy2gk,كشف مسؤول رياضي مصري أن الميدالية الذهبية التي...,1,G-7,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-3
1,Jivkt5fbyLOBsaQd,أشاد وزير الدولة للشؤون الخارجية لدولة قطر، سل...,1,G-8,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS


,missing
text,0
Qatar Related,0
Geography,0
Politics & Conflict,0
Health & Wellbeing,0
Science,0
Sports,0



Qatar Related


,count
Qatar Related,
0,2005
1,1781


...

,count
Health & Wellbeing,
NO_HEALTH,3108
H-1,349
H-2,329



Science


,count
Science,
NO_SCIENCE,2605
SC-1,588
SC-3,321
SC-2,272



Sports


,count
Sports,
NO_SPORTS,2314
SP-3,461
SP-4,370
SP-2,338
SP-1,303


In [14]:
# Clean and normalize Arabic text for both training and test sets using custom rules
train_df["processed_text"] = train_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)
test_df["processed_text"] = test_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)

# Split the dataset into 80% training and 20% validation
# We use 'stratify' on the binary 'Qatar Related' labels to maintain consistent class proportions
train_part, validation_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df[QATAR_COLUMN],
)
print(f"Training rows: {len(train_part)}; validation rows: {len(validation_part)}")

Training rows: 3028; validation rows: 758


## Gemini for `Qatar Related`

This intentionally basic prompt returns only `0` or `1`. Keep the API key in Colab Secrets or an environment variable.

### 1. Persistent Caching Mechanism
To save costs and avoid rate limits during experimentation, we use a SQLite-based disk cache. The cache key is derived from a hash of both the article text and the system prompt, ensuring that if you update your instructions, the model will re-run the classification.

In [15]:
CACHE_DB_PATH = "gemini_cache.db"

# Initialize the database
with sqlite3.connect(CACHE_DB_PATH) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS predictions (key TEXT PRIMARY KEY, val INTEGER)")

thread_local = threading.local()

def get_db_conn():
    if not hasattr(thread_local, "conn"):
        thread_local.conn = sqlite3.connect(CACHE_DB_PATH, timeout=30.0, check_same_thread=False)
    return thread_local.conn

def generate_cache_key(text, system_prompt):
    # Combine prompt and text to ensure cache is dynamic to instruction changes
    combined = f"{system_prompt}::{text}"
    return hashlib.md5(combined.encode("utf-8")).hexdigest()

def get_cached_val(text, system_prompt):
    key = generate_cache_key(text, system_prompt)
    conn = get_db_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT val FROM predictions WHERE key=?", (key,))
    row = cursor.fetchone()
    return row[0] if row else None

def set_cached_val(text, system_prompt, val):
    key = generate_cache_key(text, system_prompt)
    conn = get_db_conn()
    with conn:
        conn.execute("INSERT OR REPLACE INTO predictions VALUES (?, ?)", (key, val))

### 2. Gemini API Configuration
We define the classification rules here. The prompt is designed to handle edge cases where Qatar is mentioned incidentally versus playing an active role.

In [16]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Please add GEMINI_API_KEY to your Colab Secrets.")

client = genai_new.Client(api_key=GEMINI_API_KEY)

SYSTEM_INSTRUCTION = """
You are an expert Arabic news classification assistant.
Your task is to determine if the given article is related to Qatar.

Rules:
- Output 1 if Qatar (government, economy, sports, diplomacy, residents, or events) plays a meaningful role.
- Output 1 even if other entities are mentioned, as long as Qatar's involvement is significant.
- Output 0 ONLY if Qatar is absent, a passing location in a list, or an incidental historical mention.

Respond ONLY with a single digit: 0 or 1.
"""

### 3. Classification Logic and Execution
We use a `ThreadPoolExecutor` to speed up processing of the dataset. Results are stored in the validation and test DataFrames.

In [17]:
def classify_qatar(text):
    cached_val = get_cached_val(text, SYSTEM_INSTRUCTION)
    if cached_val is not None:
        return cached_val

    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL_NAME,
            contents=f"Article:\n{text}",
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                temperature=0.0,
            ),
        )
        output = response.text.strip() if response.text else ""
        match = re.search(r"\b([01])\b", output)
        val = int(match.group(1)) if match else 0

        set_cached_val(text, SYSTEM_INSTRUCTION, val)
        return val
    except Exception as e:
        print(f"API Error: {e}")
        return 0

def run_parallel_predictions(texts):
    with ThreadPoolExecutor(max_workers=10) as executor:
        return list(executor.map(classify_qatar, texts))

# Run on Validation
print("Processing Validation Set...")
val_preds = run_parallel_predictions(validation_part[TEXT_COLUMN].tolist())
validation_part[QATAR_COLUMN] = val_preds

# Run on Test
print("Processing Test Set...")
test_preds = run_parallel_predictions(test_df[TEXT_COLUMN].tolist())
test_df[QATAR_COLUMN] = test_preds

Processing Validation Set...
Processing Test Set...


## AraBERT for the remaining targets

This model fine-tunes **5 output heads** (Geography, Politics & Conflict, Health & Wellbeing, Science, and Sports). The 'Qatar Related' classification is handled separately by Gemini. By focusing AraBERT on these specific categories, we leverage the strengths of both models.

### 1. Model Initialization and Label Encoding
We define the base AraBERT model for our encoder and initialize `LabelEncoder` objects for the five categorical targets. Note that `Qatar Related` is excluded here as it is handled by the Gemini LLM.

In [27]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Initialize encoders for non-Qatar targets defined in REMAINING_TARGETS
label_encoders = {
    column: LabelEncoder().fit(train_df[column]) for column in REMAINING_TARGETS
}

### 2. Dataset Preparation
In this step, we wrap our training and validation splits into the custom `NewsDataset` class. This handles tokenization, padding, and mapping labels to the correct tensor formats for the multi-head heads.

In [28]:
# Prepare PyTorch Datasets for the model
arabert_train = NewsDataset(train_part)
arabert_validation = NewsDataset(validation_part)

### 3. Multi-Head Training
We initialize the `MultiHeadAraBERT` model with specific head counts for each target and use the Hugging Face `Trainer` to fine-tune the model. We use a custom data collator to correctly batch the multiple label dictionaries.

In [29]:
# Map class counts to define output dimensions for each linear head
class_counts = {col: len(label_encoders[col].classes_) for col in REMAINING_TARGETS}

# Initialize model and training configuration
arabert_model = MultiHeadAraBERT(class_counts)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "arabert"),
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=arabert_model,
    args=training_args,
    train_dataset=arabert_train,
    eval_dataset=arabert_validation,
    data_collator=custom_data_collator,
)

# Execute fine-tuning
print("Starting AraBERT multi-head training...")
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting AraBERT multi-head training...


Epoch,Training Loss,Validation Loss
1,No log,0.362585


TrainOutput(global_step=379, training_loss=0.5382349019314808, metrics={'train_runtime': 162.0299, 'train_samples_per_second': 18.688, 'train_steps_per_second': 2.339, 'total_flos': 0.0, 'train_loss': 0.5382349019314808, 'epoch': 1.0})

### 4. Hybrid Validation Evaluation
We evaluate the model's performance on the validation split. By combining the zero-shot LLM predictions for 'Qatar Related' with the fine-tuned AraBERT predictions for the remaining 5 targets, we get a complete picture of our multi-label accuracy.

In [30]:
# Generate AraBERT predictions for the validation split
arabert_val_results = predict_arabert(arabert_model, validation_part, is_test=False)

# Integrate Gemini's Qatar predictions
arabert_val_results[QATAR_COLUMN] = validation_part[QATAR_COLUMN].values

# Calculate scores across all 6 targets
val_scores = score_predictions(validation_part[TARGET_COLUMNS], arabert_val_results)

print("--- Hybrid Model Validation Performance ---")
for target, f1 in val_scores.items():
    print(f"{target:20}: {f1:.4f}")

--- Hybrid Model Validation Performance ---
Qatar Related       : 1.0000
Geography           : 0.7413
Politics & Conflict : 0.8941
Health & Wellbeing  : 0.9351
Science             : 0.9234
Sports              : 0.9077
mean                : 0.9003


### 5. Test Set Inference
Now we apply the AraBERT model to the competition test set to classify the 5 general categories.

In [31]:
# Run AraBERT inference on test data
print("Generating AraBERT predictions for test set...")
arabert_test_results = predict_arabert(arabert_model, test_df, is_test=True)

Generating AraBERT predictions for test set...


### 6. Final Submission Construction
We merge the results from both models and save the final CSV. This ensures the output format matches the ZINDI competition requirements.

In [32]:
# Merge Gemini Qatar predictions into the final test result dataframe
arabert_test_results[QATAR_COLUMN] = test_df[QATAR_COLUMN].values

# Create submission file
final_sub_path = OUTPUT_DIR / "submission_llm_arabert.csv"
submission_df = make_submission(test_df, arabert_test_results, final_sub_path)

print(f"Submission saved to: {final_sub_path}")
display(submission_df.head())

Submission saved to: /content/submissions/submission_llm_arabert.csv


,ID,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,jiRadWgPKa2i51Xl,1,G-3,NO_POLITICS,NO_HEALTH,SC-3,NO_SPORTS
1,IoKhm6KjwCkwEnAZ,1,G-4,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
2,ZihSiR1hnDU6w7Lz,1,G-3,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
3,X72YJR7IVU7CCKbh,1,G-4,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-4
4,TpxFAGsHEekF2MpS,0,G-11,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS
